# 1-D spherical TI, frequency domain — P-SV (spheroidal) with flexOPT, fluid core in $Q=P/\omega$

Companion of `SphericalTI_SH_oct2026.ipynb`, same step-by-step workflow (each step = stop-and-review point):

| step | content |
|---|---|
| **P1** | symbolic: TI elastic equation → normalised spheroidal fields $\mathbf u=U\,\mathbf P_{lm}+V\,\mathbf B_{lm}$ → projection onto $\mathbf P^*_{lm},\mathbf B^*_{lm}$ → two coupled radial equations; fluid equation in $Q=P/\omega$ (Geller & Ohminato 1994); two `famousEquation` entries — **this step** |
| P2 | FD3/OPT3/OPT4/OPT5 recipes (2 fields, scaled coordinates, CPU), operators vs conventional DSM P-SV on planet1D models |
| P3 | solid shell traction-free at both ends: exact spheroidal eigenfrequencies; native half-element closures |
| P4 | fluid outer core: G&O solid–fluid coupling terms ($+\omega w_iQn_i^{(S)}$, $-\omega w\,u_in_i^{(F)}$) at CMB/ICB as interface rows |
| P5 | seismograms, $f_z=1$ and $f_y=1$ at the pole, $u_r,u_\theta$ on φ = 0 |

Conventions: $Y_{lm}$ fully normalised, $\mathbf P_{lm}=\hat{\mathbf r}Y_{lm}$, $\mathbf B_{lm}=k^{-1}\nabla_1Y_{lm}$, $k=\sqrt{l(l+1)}$ (with $\mathbf C_{lm}$ of the SH notebook the three are orthonormal); $x\equiv r$.
Gravity: not included (to be discussed separately).

In [ ]:
# Same bootstrap as seismo1Dbenchmark.ipynb
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()

# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")

include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .commonBatchs, .planet1D, .flexOPT

using Symbolics, LinearAlgebra, Random
include(joinpath(flexopt_root, "helpers", "sphericalTI_SH.jl"))        # shared: TI tensor, spherical divergence, jet
include(joinpath(flexopt_root, "helpers", "sphericalTI_PSV.jl"))

## P1a — projection of the TI operator on the spheroidal harmonics

Zonal jet as for SH ($q=\cot\theta$, $p_0=P$, $p_1=P'$, $P''=-qP'-k^2P$), $\mathbf u=(U p_0,\;V p_1/k,\;0)$.
Expected: $E_r=R_U\,p_0$, $E_\theta=R_V\,p_1/k$, $E_\phi=0$, with (traction fluxes taken out)

$$r^2R_U=\partial_r\big(r^2C U'+2rFU-krFV\big)-2rFU'-krLV'+(\omega^2\rho r^2-4A+4N-k^2L)\,U+k(2A+L-2N)\,V$$
$$r^2R_V=\partial_r\big(r^2LV'-rLV+krLU\big)+krFU'+rLV'+(\omega^2\rho r^2-k^2A-L+2N)\,V+k(2A+L-2N)\,U$$

All five TI parameters enter; no material derivative and no second derivative outside the fluxes; the natural weak form is symmetric (DSM's $\omega^2T-H$).

In [ ]:
@show checkTIisotropicLimitAndLoveNotation()
symPSV = deriveSpheroidalSymbolic()
println("E_φ = ", symPSV.E[3])
println("\nR_U =\n", symPSV.RU)
println("\nR_V =\n", symPSV.RV)

In [ ]:
# numerical verification at random (r, cotθ, P, P') for several l (relative to |E|)
for l in (1, 2, 5, 20)
    @show l verifySpheroidalProjection(; l=l)
end

### Independent check with $m\ne0$ — full Cartesian TI operator
$\mathbf u=U(r)\,Y\hat{\mathbf r}+V(r)\nabla_1Y/k$ for $Y\propto xz/r^2$ (l = 2, m = 1) and $Y\propto(x^2-y^2)z/r^3$ (l = 3, m = 2),
compared with $R_UY\hat{\mathbf r}+R_V\nabla_1Y/k$ at random points (SymPy prototype: agreement to $10^{-137}$).

In [ ]:
@time @show cartesianSpheroidalCheck(; harmonic=:Y21)
@time @show cartesianSpheroidalCheck(; harmonic=:Y32)

## P1b — fluid regions: $Q=P/\omega$ (Geller & Ohminato 1994, §3)

$Q=\lambda\nabla\cdot\mathbf u/\omega$, strong form $\omega^2Q/\lambda+\partial_i(Q_{,i}/\rho)=0$ (G&O eq. 28/29), weak form
$\int[\omega^2wQ/\lambda-w_{,i}Q_{,i}/\rho]\,dV$ (eq. 31). With $Q=q(r)Y_{lm}$:

$$r^2\Big[\frac{\omega^2q}{\lambda}+\frac1{r^2}\Big(\frac{r^2q'}{\rho}\Big)'-\frac{\Lambda q}{\rho r^2}\Big]=\partial_r\Big(\frac{r^2}{\rho}\,q'\Big)+\Big(\frac{\omega^2r^2}{\lambda}-\frac{\Lambda}{\rho}\Big)q .$$

Continuity: $Q$ essential, $Q_{,i}n_i/\rho$ (normal displacement) natural — the opposite of the solid. Solid–fluid coupling (Step P4) only through the surface
terms of G&O eq. (35): $+\int\omega w_iQn^{(S)}_i\,dS$ in the solid rows, $-\int\omega w\,u_in^{(F)}_i\,dS$ in the fluid rows ⇒ $(\omega^2T-H+\omega R')c=-g$ (eq. 46).

In [ ]:
for l in (1, 3, 20)
    @show l verifyFluidProjection(; l=l)
end

## P1c — `famousEquation` entries

* `"1DsphericalPSV_TI_Freq"` — fields $(U,V)$, geometric material fields (per $l$; the recipe stays $l$-independent):
  $\rho r^2,\;Cr^2,\;Fr,\;Lr^2,\;Lr,\;kFr,\;kLr$, and $m_{UU}=-4A+4N-\Lambda L$, $m_{VV}=-\Lambda A-L+2N$, $m_{UV}=k(2A+L-2N)$.
* `"1DsphericalFluidQ_Freq"` — field $q$, fields $r^2/\rho,\;r^2/\lambda,\;\Lambda/\rho$.

Their natural weak forms should show the traction fluxes $r^2\sigma_{rr}$, $r^2\sigma_{r\theta}$ (solid) and $r^2q'/\rho$ (fluid) as boundary fluxes.

In [ ]:
eqPSV, forcePSV = famousEquations("1DsphericalPSV_TI_Freq")
@show eqPSV.fields eqPSV.vars forcePSV.exprs
wfPSV = naturalWeakForm(eqPSV)
for (ie, name) in enumerate(("U-equation", "V-equation"))
    println("\n$name — volume terms (test derivative order => expression):")
    for t in wfPSV.volume_terms[ie]
        println("   ", t.test_derivative_orders, "  =>  ", t.expression)
    end
    println("$name — boundary fluxes:")
    for b in wfPSV.boundary_fluxes[ie]
        println("   ", b.normal_coordinate, "  =>  ", b.expression)
    end
end
@show verifyPSVEntry(; l=3) verifyPSVEntry(; l=10)

In [ ]:
eqQ, forceQ = famousEquations("1DsphericalFluidQ_Freq")
@show eqQ.exprs eqQ.vars
wfQ = naturalWeakForm(eqQ)
for t in wfQ.volume_terms[1]; println("   ", t.test_derivative_orders, "  =>  ", t.expression); end
for b in wfQ.boundary_fluxes[1]; println("   flux ", b.normal_coordinate, "  =>  ", b.expression); end

## ⏸ Stop and review (end of P1)
1. `verifySpheroidalProjection`: discrepancies at round-off for all l; `cartesianSpheroidalCheck`: ~1e-15 relative for both harmonics;
2. `verifyFluidProjection`: round-off;
3. weak forms: solid fluxes = $-(r^2\sigma_{rr})$, $-(r^2\sigma_{r\theta})$ split into their terms, fluid flux $-r^2q'/\rho$; `verifyPSVEntry` round-off.

Next (P2): two-field recipes (FD3/OPT3/OPT4/OPT5, scaled coordinates as for SH: $Cr^2,Lr^2$ unchanged, first-order fields × h, zeroth-order × h²)
and the conventional DSM P-SV matrices as reference.